# 第5日 ノートブック03：YOLO と PatchCore を比べる / Day 5 Notebook 03: YOLO versus PatchCore

---

<!-- taj-unit: day-5-notebook-03-yolo-versus-patchcore -->

## 学習目標 / Learning objectives

1. 2 つの手法の出力の違いを整理できる<br>Summarise how the two methods differ in output <!-- taj-unit: summarise-how-the-two-methods-di -->
2. 同じ画像セットで両方の結果を並べられる<br>Line up both results on the same image set <!-- taj-unit: line-up-both-results-on-the-same -->
3. 同じ基準で OK/NG を比べるときの注意を説明できる<br>Explain the caution needed when comparing OK/NG decisions <!-- taj-unit: decisions -->
4. それぞれの長所・短所を製造の観点で説明できる<br>Explain strengths and weaknesses from a manufacturing view <!-- taj-unit: explain-strengths-and-weaknesses -->
5. 塩ビ検査にどちらを、どの順序で使うか提案できる<br>Propose which to use and in what order for PVC <!-- taj-unit: propose-which-to-use-and-in-what -->

## このノートブックの基本情報 / Notebook facts

| 項目 / Item | 内容 / Details |
| --- | --- |
| 学習時間（目安） / Estimated duration | 100 分 / 100 minutes <!-- taj-row: duration --> |
| 難易度 / Difficulty | 初級 / Beginner <!-- taj-row: difficulty --> |
| 前提条件 / Prerequisites | 第4日ノートブック03 と第5日ノートブック02 を実行し、`submissions` に結果が保存されていること。 / You ran Day 4 Notebook 03 and Day 5 Notebook 02, so the results exist in `submissions`. <!-- taj-row: prerequisites --> |
| 必要なライブラリ / Required libraries | pandas, numpy, matplotlib, pillow <!-- taj-row: libraries --> |
| 使用カーネル / Jupyter kernel | `aivision-anomalib`（YOLO の結果はファイルから読み込みます） <!-- taj-row: kernel --> |
| この日の時間割 / Today's timetable | [`COURSE_SYLLABUS.md` 第5日 / Day 5](../../COURSE_SYLLABUS.md#day5) <!-- taj-row: timetable --> |

<!-- taj-unit: notebook-facts -->

### 提出物 / Expected deliverables

- 比較表<br>The comparison table <!-- taj-unit: the-comparison-table -->
- 並べた結果画像<br>Side-by-side result images <!-- taj-unit: side-by-side-result-images -->
- 塩ビ検査への推奨（判断と理由）<br>A recommendation for PVC with reasons <!-- taj-unit: a-recommendation-for-pvc-with-re -->

> YOLO and PatchCore live in two different environments, so this notebook compares the **saved results** instead of loading both models. That is a deliberate design choice, documented in `requirements/tested_versions.md`.


## このノートブックの位置 / Where this notebook sits

| 項目 / Item | 内容 / Details |
| --- | --- |
| 日 / Day | 第5日 / Day 5 <!-- taj-row: day --> |
| この日の順番 / Position today | 3 / 3（この日のノートブックのうち）/ of today's notebooks <!-- taj-row: today --> |
| 所要時間 / Duration | 100 分 / 100 minutes <!-- taj-row: duration --> |
| 前提 / Prerequisites | `Day04_03` と `Day05_02` の保存済み結果 <!-- taj-row: prerequisites --> |
| この日の時間割 / Today's timetable | [`COURSE_SYLLABUS.md` 第5日 / Day 5](../../COURSE_SYLLABUS.md#day5) <!-- taj-row: timetable --> |

1 日の時間割は `COURSE_SYLLABUS.md` に 1 つだけあります。このノートブックはその一部で、
自分の時間割を持ちません。時刻が分からなくなったら、まずそこを見てください。

The day's timetable exists once, in `COURSE_SYLLABUS.md`. This notebook is one part of it and
does not keep its own schedule; look there first when you are unsure of the time.

**ここは 2 回開きます。** ステップ 3 は較正と `clarity-freeze-record` まで。ステップ 4 は [Day04_03](../Day04_YOLO_Training/Day04_03_YOLO_Evaluation.ipynb) の `shared-sealed-test`、ステップ 5 は [Day05_02](Day05_02_PatchCore_Training.ipynb) の `clarity-sealed-test-scoring` です。両方を封印したら、ステップ 6 で `clarity-load-sealed-test` から再開して比較表を作ります。

**Open this notebook twice.** At step 3, use calibration and stop after `clarity-freeze-record`. Run `shared-sealed-test` in [Day04_03](../Day04_YOLO_Training/Day04_03_YOLO_Evaluation.ipynb) at step 4, then `clarity-sealed-test-scoring` in [Day05_02](Day05_02_PatchCore_Training.ipynb) at step 5. Once both seals exist, return here for step 6: resume at `clarity-load-sealed-test` and compute the comparison. This notebook runs no TEST inference.

<!-- taj-unit: where-this-notebook-sits -->


### 予想して、実行して、説明する / Predict, run, inspect, explain

較正は判定条件を決める工程です。試験は固定後に別データで確認する工程です。 重要なセルの前に、入力・操作・出力を一文ずつ説明してください。
予想を書いて実行し、値または図を観察します。較正用スライダーだけを変え、もう一度予想して実行します。
意図的なエラーと未完成の演習は、その指示に従って取り組みます。

Calibration selects the rule; test checks it on different data after freezing. Before an important cell, describe its input, operation and output in one sentence each.
Predict, run, inspect the values/image and explain why. Change only the calibration slider, predict and rerun.
Follow the instructions for deliberately erroneous cells and unsolved exercises.

<!-- taj-unit: predict-run-inspect-explain -->


---


## Part 0　準備 / Preparation

<!-- taj-unit: preparation -->


In [ ]:
# ▸ 実行のみ / Run only：このまま実行します。書き換えません。
# ▸ Run only: press Shift + Enter and leave the code as it is.
from pathlib import Path
import sys

COURSE_ROOT = Path.cwd().resolve()
while not ((COURSE_ROOT / "assets").is_dir() and (COURSE_ROOT / "notebooks").is_dir()):
    if COURSE_ROOT.parent == COURSE_ROOT:
        raise FileNotFoundError(
            "研修フォルダが見つかりません。JupyterLab を AI_Vision_Training で起動してください。\n"
            "Course folder not found. Start JupyterLab inside the AI_Vision_Training folder."
        )
    COURSE_ROOT = COURSE_ROOT.parent

ASSETS = COURSE_ROOT / "assets"
sys.path.insert(0, str(ASSETS / "helpers"))
import aivision

print("研修フォルダ / course root :", COURSE_ROOT)
print("素材フォルダ / assets      :", ASSETS)

In [ ]:
# ▸ 実行のみ / Run only：このまま実行します。書き換えません。
# ▸ Run only: press Shift + Enter and leave the code as it is.
import importlib
import sys

print("Python version:", sys.version.split()[0])

REQUIRED = [
    ("numpy", "numpy"),
    ("matplotlib", "matplotlib"),
    ("pandas", "pandas"),
    ("PIL", "pillow"),
]

missing = []
for import_name, pip_name in REQUIRED:
    try:
        module = importlib.import_module(import_name)
        version = getattr(module, "__version__", "unknown")
        print(f"  [OK]      {pip_name:<20} {version}")
    except ImportError:
        print(f"  [MISSING] {pip_name:<20} -> pip install {pip_name}")
        missing.append(pip_name)

if missing:
    print("\n未導入のパッケージがあります / Some packages are missing:", ", ".join(missing))
    print("docs/troubleshooting/installation.md を確認してください / see that file for help.")
else:
    print("\nこのノートブックに必要なライブラリはすべて揃っています。")
    print("All libraries required by this notebook are available.")

In [ ]:
# ▸ 実行のみ / Run only：このまま実行します。書き換えません。
# ▸ Run only: press Shift + Enter and leave the code as it is.
FONT_FAMILY = aivision.configure_japanese_font()
aivision.set_plot_style()

if FONT_FAMILY is None:
    print("日本語フォントが見つかりません。図のタイトルは英字になります。")
    print("No Japanese font found; figure titles fall back to English.")
else:
    print("図の日本語フォント / figure font:", FONT_FAMILY)

In [ ]:
# ▸ 実行のみ / Run only：このまま実行します。書き換えません。
# ▸ Run only: press Shift + Enter and leave the code as it is.
import json
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt

def show(images, titles=None, figsize=(13, 4.5)):
    if not isinstance(images, (list, tuple)):
        images = [images]
    if titles is None:
        titles = [""] * len(images)
    fig, axes = plt.subplots(1, len(images), figsize=figsize)
    if len(images) == 1:
        axes = [axes]
    for ax, image, title in zip(axes, images, titles):
        ax.imshow(image)
        ax.set_title(title, fontsize=9)
        ax.set_xticks([]); ax.set_yticks([])
    plt.tight_layout()
    plt.show()

---


## Part 1　出力の違いをおさらいする / Recap: the outputs differ

<!-- taj-unit: recap-the-outputs-differ -->


In [ ]:
# ▸ 実行のみ / Run only：このまま実行します。書き換えません。
# ▸ Run only: press Shift + Enter and leave the code as it is.
show([Image.open(ASSETS / "diagrams" / "diagram_yolo_vs_patchcore.png")],
     ["two different answers to two different questions"], figsize=(11, 5.5))

#### 同じにできないものは、同じに扱わない / Do not pretend different things are the same

2 つの手法は、**返すものが違います**。

The two methods **return different things**:

|  | YOLO | PatchCore |
| --- | --- | --- |
| 主な出力<br>main output | 四角 + クラス + confidence<br>box + class + confidence | 異常スコア + ヒートマップ<br>anomaly score + heatmap <!-- taj-row: main-output --> |
| 「何の欠陥か」<br>"which defect" | **分かる**<br>**yes** | 分からない<br>no <!-- taj-row: which-defect --> |
| 「どこか」<br>"where" | 四角の位置<br>box location | ヒートマップの明るい場所<br>bright regions of the heatmap <!-- taj-row: where --> |
| 学習に要るもの<br>fitting needs | 欠陥のラベル<br>defect labels | 正常と確認できた画像<br>images confirmed as normal <!-- taj-row: fitting-needs --> |

したがって、**mAP と AUROC を直接比べてはいけません**。
測っているものが違うからです。

Therefore **mAP and AUROC must not be compared directly** — they measure different things.

比べてよいのは、**同じ正解ラベルに対して、それぞれの手法が画像単位で出した OK/NG の判定** です。
そのためには、**それぞれの手法に判定規則（しきい値）を宣言し、両方を同じ正解と突き合わせます**。
2 つの手法どうしが一致しているかだけを見ると、**両方が同じように間違っている場合** を見逃します。

What *can* be compared is **each method's image-level OK/NG decision against the same ground
truth**, once **each method has a declared decision rule (threshold)**. Comparing only whether the
two methods agree would hide the case where **both are wrong in the same way**.

<!-- taj-unit: do-not-pretend-different-things-are-the-same -->


---


## Part 2　保存された結果を読み込む / Loading the saved results

<!-- taj-unit: loading-the-saved-results -->


### 比較の主な流れ / The primary comparison protocol

![較正後に全設定を固定して試験 / calibrate, freeze all settings, then test](../../assets/svg/teach_frozen_threshold_pipeline.svg)

図：固定前の較正と固定後の試験を区別する模式図です。この授業では次の6段階に対応させます。

Figure: this schematic separates calibration from final test and maps to six concrete classroom stages:

<!-- taj-unit: the-primary-comparison-protocol-caption -->

**気づいてほしいこと / What to notice**

1. 分割manifestと同じ画像IDを確認 / check the split manifest and identical image IDs.
2. 両モデルの予測をIDで対応 / pair predictions by image ID.
3. 較正だけでYOLO CとPatchCore Tを選ぶ / select YOLO C and PatchCore T on calibration only.
4. モデル・前処理・正常メモリ・後処理・しきい値を記録し固定 / record and freeze models, preprocessing, memory, postprocessing and thresholds.
5. 未使用testを一度評価 / assess untouched test once.
6. 同じ画像単位でTP/FP/FN/TNを比較 / compare TP/FP/FN/TN in the same image units.

| 分割 / Split | 許される操作 / Allowed action |
| --- | --- |
| fit | YOLO重み・PatchCore正常メモリを作る / build YOLO weights and PatchCore normal memory <!-- taj-row: fit --> |
| calibration | 設定・しきい値を選ぶ / choose settings and thresholds <!-- taj-row: calibration --> |
| test | 固定後の評価だけ、調整禁止 / assessment after freeze only; no tuning <!-- taj-row: test --> |

較正は正常n=6・異常n=20の保存例です。**考えてみてください / Try to answer**

予想：誤報率上限10%なら正常6枚で何枚まで？
`floor(6×0.10)=0`件です。本番で10%以下という保証ではありません。
同梱図の0.06/0.50は保存例で、今回のしきい値は較正コードの計算値を使います。

The saved calibration example has n=6 normal and n=20 abnormal images.
Predict the allowed false rejects at a 10% cap: `floor(6×0.10)=0`.
This is not a production-rate guarantee. Supplied 0.06/0.50 values are archival examples;
use thresholds computed by your calibration code.

<!-- taj-unit: what-to-notice -->


#### 較正用の 3 つのファイルを読む / Read the three calibration files

| ファイル<br>File | 作った場所<br>Produced by | 何が入っているか<br>Contents |
| --- | --- | --- |
| `submissions/day05/split_manifest.json` | 第5日ノートブック02<br>Day 5 Notebook 02 | **分割の記録**。第4日が決めた分割と一致することを確かめた印つき<br>**the split record**, stamped as matching the split Day 4 decided <!-- taj-row: split-manifest --> |
| `submissions/day04/yolo_predictions_on_shared_set.json` | 第4日ノートブック03<br>Day 4 Notebook 03 | **較正用だけ** の YOLO 検出（信頼度つき）<br>YOLO detections with confidences, **calibration only** <!-- taj-row: yolo-predictions-shared-set --> |
| `submissions/day05/patchcore_predictions.json` | 第5日ノートブック02<br>Day 5 Notebook 02 | **較正用だけ** の PatchCore スコア<br>PatchCore scores, **calibration only** <!-- taj-row: patchcore-predictions --> |

**分割の記録が無い結果は使いません。** どのデータでしきい値を決めたか分からない数字は、比較の材料に
なりません。古い形式のファイルは、理由を表示して止めます。

**A result without its split record is not used.** A number whose calibration data is unknown cannot
support a comparison. Outdated files stop the notebook with an explanation.

試験用のスコアは、この Part ではまだ読みません。**固定を記録したあと** でだけ、
`yolo_predictions_sealed_test.json` と `patchcore_test_predictions.json` を読み込みます。

The test scores are not read in this Part. Only **after the freeze is recorded** does the notebook
read `yolo_predictions_sealed_test.json` and `patchcore_test_predictions.json`.

<!-- taj-unit: read-the-three-calibration-files -->


In [ ]:
# ▸ この値を変更 / Change this value：値を変えて、結果を予想してから実行します。
# ▸ Change this value: edit it below, predict the result, then run.
import json
from pathlib import Path

manifest_path = COURSE_ROOT / "submissions" / "day05" / "split_manifest.json"
yolo_path = COURSE_ROOT / "submissions" / "day04" / "yolo_predictions_on_shared_set.json"
patchcore_path = COURSE_ROOT / "submissions" / "day05" / "patchcore_predictions.json"

REQUIRED_SCHEMA = 3
missing = [p for p in (manifest_path, yolo_path, patchcore_path) if not p.is_file()]
if missing:
    for path in missing:
        print("見つかりません / missing:", path)
    print()
    print("先に実行してください / run first:")
    print("  notebooks/Day04_YOLO_Training/Day04_03_YOLO_Evaluation.ipynb")
    print("  notebooks/Day05_PatchCore/Day05_02_PatchCore_Training.ipynb")
    raise FileNotFoundError(
        "比較に必要なファイルがそろっていません / the files needed for the comparison are missing"
    )

manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
yolo_data = json.loads(yolo_path.read_text(encoding="utf-8"))
patchcore_data = json.loads(patchcore_path.read_text(encoding="utf-8"))

stale = []
for name, payload in (("split_manifest.json", manifest),
                      ("yolo_predictions_on_shared_set.json", yolo_data),
                      ("patchcore_predictions.json", patchcore_data)):
    version = payload.get("schema_version")
    if version != REQUIRED_SCHEMA:
        stale.append(f"{name}: schema_version={version!r}（必要 / required: {REQUIRED_SCHEMA}）")
if stale:
    print("古い形式の結果ファイルです / these result files are from an older method:")
    for line in stale:
        print("  -", line)
    print()
    print("古い比較結果を上書きして「新しい方法で出した数字」として扱うことはできません。")
    print("第4日ノートブック03 と第5日ノートブック02 を実行し直してください。")
    print("Old comparisons must not be overwritten and passed off as new results.")
    print("Re-run Day 4 Notebook 03 and Day 5 Notebook 02.")
    raise ValueError("stale result files / 古い結果ファイル")

# 分割は第4日が「推論より前」に決め、第5日ノートブック02 が anomalib と照合した記録です。
# The split was decided by Day 4 before any inference and verified against anomalib by Notebook 02.
shared_split_record = manifest.get("shared_split") or {}
if not shared_split_record.get("verified"):
    raise ValueError(
        "分割記録が第4日の比較用セットと照合されていません。第5日ノートブック02 を実行し直してください / "
        "the split manifest was not verified against the Day 4 shared split; re-run Notebook 02")
partitions_digest = shared_split_record.get("partitions_digest")

for name, payload in (("yolo_predictions_on_shared_set.json", yolo_data),
                      ("patchcore_predictions.json", patchcore_data)):
    if payload.get("test_scored"):
        raise ValueError(
            f"{name} は試験用を含んでいます。固定前に採点しないでください / "
            f"{name} already contains test scores; do not score test before the freeze")
    if payload.get("scored_partitions") != ["calibration"]:
        raise ValueError(
            f"{name} は較正用だけのファイルではありません / {name} must hold the calibration "
            f"partition only, found {payload.get('scored_partitions')!r}")
    calibration_names = set(manifest["partitions"]["calibration"])
    outside = sorted(row.get("image") for row in payload.get("images") or []
                     if row.get("partition") != "calibration"
                     or row.get("image") not in calibration_names)
    if outside:
        raise ValueError(
            f"{name} に較正用以外の行があります / non-calibration rows in {name}: {outside[:5]}")

if yolo_data.get("split_partitions_digest") != partitions_digest:
    raise ValueError(
        "YOLO の較正用予測が、この分割のものではありません。第4日ノートブック03 の Part 5 を"
        "実行し直してください / the YOLO calibration predictions belong to another split; re-run "
        "Part 5 of Day 4 Notebook 03")

# YOLO の重みは、名前ではなく中身で確かめます。同じパスで学習し直した best.pt は別の実験です。
# Verify the YOLO weights by content, not by name: a retrained best.pt at the same path is another
# experiment, and its scores must not be compared as if they came from the frozen model.
from experiment_identity import ProvenanceError, check_file_sha256

weights_path = yolo_data.get("weights")
if not weights_path:
    raise ValueError(
        "YOLO の較正用予測に重みのパスがありません。第4日ノートブック03 の Part 5 を実行し直して"
        "ください / the YOLO calibration predictions record no weights path; re-run Part 5 of "
        "Day 4 Notebook 03")
try:
    check_file_sha256(COURSE_ROOT / weights_path, yolo_data.get("weights_sha256"),
                      what="YOLO の重み / the YOLO weights")
except ProvenanceError as error:
    raise ValueError(
        "YOLO の重みの中身が、較正用の予測を出したときと違います。同じ重みで採点し直してください / "
        "the YOLO weights differ from the ones that produced the calibration scores; score again "
        "with the recorded weights.\n" + str(error)) from error

print("分割の件数 / partition sizes:", manifest["counts"])
print("分割の重なり / overlap check  :", manifest["overlap_check"])
print("スコアの決まり / score convention:", manifest["score_convention"][:70], "...")

#### フレームにする / Turn them into frames

`image` 列はフォルダをまたいで同じ書き方にそろえます（`画像の種類/ファイル名.png`）。そろえないと、同じ画像を 2 つの手法で結び付けられません。

The `image` column is normalised to `type/file.png` for both methods. Without that, the same image cannot be matched across the two methods.

<!-- taj-unit: turn-them-into-frames -->


In [ ]:
# ▸ 実行のみ / Run only：このまま実行します。書き換えません。
# ▸ Run only: press Shift + Enter and leave the code as it is.
import pandas as pd

yolo_frame = pd.DataFrame(yolo_data["images"])
patchcore_frame = pd.DataFrame(patchcore_data["images"])

yolo_frame["image"] = yolo_frame["image"].astype(str)
patchcore_frame["image"] = patchcore_frame["image"].astype(str).str.replace("^test/", "", regex=True)
yolo_frame["image"] = yolo_frame["image"].str.replace("^test/", "", regex=True)

partition_of = {}
for partition, files in manifest["partitions"].items():
    for name in files:
        partition_of[name.replace("test/", "", 1)] = partition

yolo_frame["truly_abnormal"] = yolo_frame["true_label"].astype(str).eq("bad")
yolo_frame["partition"] = yolo_frame["image"].map(partition_of)
patchcore_frame["partition"] = patchcore_frame["image"].map(partition_of)

print("YOLO の行数 / YOLO rows          :", len(yolo_frame))
print("PatchCore の行数 / PatchCore rows:", len(patchcore_frame))
print("分割が決まらない行 / rows without a partition:",
      int(yolo_frame["partition"].isna().sum()), int(patchcore_frame["partition"].isna().sum()))
print()
print("推論した最小 confidence / minimum inference confidence:",
      yolo_data.get("min_inference_conf"))
print("YOLO の判定規則 / YOLO decision rule:", yolo_data.get("decision_rule"))

## Part 3　同じ画像で並べる / Lining them up on the same images

<!-- taj-unit: lining-them-up-on-the-same-images -->


#### 同じ画像で、両方の結果がそろっている画像だけを使う / Use only images where both methods produced a result

片方の手法しか結果が無い画像は、**比べられません**。黙って捨てると「都合のよい画像だけの比較」になります。だから、まず **何枚そろっていて、何枚足りないか** を数えて表示します。

An image with only one method's result **cannot be compared**. Dropping it silently would leave a comparison of convenient images only. So count and show **how many images match and how many are missing** before comparing anything.

<!-- taj-unit: use-only-images-where-both-methods-produced-a-result -->


In [ ]:
# ▸ 実行のみ / Run only：このまま実行します。書き換えません。
# ▸ Run only: press Shift + Enter and leave the code as it is.
REQUIRED_COLUMNS = ["partition", "image", "truly_abnormal"]

# 部分的な比較で先に進みたい場合だけ True にします。その結果は「最終」ではありません。
# Set this to True only to explore a partial subset; such a result is never final.
ALLOW_PARTIAL_COMPARISON = False


def check_unique_ids(frame, name):
    if frame["image"].duplicated().any():
        duplicated = sorted(frame.loc[frame["image"].duplicated(), "image"])[:5]
        raise ValueError(f"{name}: 同じ画像が複数行あります / duplicate image IDs: {duplicated}")
    if frame["partition"].isna().any():
        raise ValueError(f"{name}: 分割が決まらない行があります / rows without a partition")
    return set(frame["image"])


problems = []
for name, frame in (("YOLO", yolo_frame), ("PatchCore", patchcore_frame)):
    absent = [c for c in REQUIRED_COLUMNS if c not in frame.columns]
    if absent:
        problems.append(f"{name}: 列がありません / missing columns {absent}")
if problems:
    print("結果ファイルの形が想定と違います / unexpected result-file shape:")
    for line in problems:
        print("  -", line)
    print("第4日ノートブック03 と第5日ノートブック02 を実行し直してください。")
    raise ValueError("unexpected result-file shape")

yolo_ids = check_unique_ids(yolo_frame, "YOLO")
patchcore_ids = check_unique_ids(patchcore_frame, "PatchCore")

manifest_calibration = {name.replace("test/", "", 1) for name in manifest["partitions"]["calibration"]}
patchcore_calibration = set(patchcore_frame["image"])

missing_yolo = sorted(manifest_calibration - yolo_ids)
missing_patchcore = sorted(manifest_calibration - patchcore_calibration)
unknown_patchcore = sorted(patchcore_calibration - manifest_calibration)
wrong_partition = sorted(
    patchcore_frame.loc[patchcore_frame["partition"] != "calibration", "image"])

print("較正用の画像 / calibration images in the manifest:", len(manifest_calibration))
print("YOLO の較正用の行 / YOLO calibration rows        :", len(yolo_frame[yolo_frame["partition"] == "calibration"]))
print("PatchCore の較正用の行 / PatchCore calibration rows:", len(patchcore_frame))
print()
print("YOLO が欠けている較正用画像 / calibration images without YOLO   :", len(missing_yolo))
print("PatchCore が欠けている較正用画像 / without PatchCore          :", len(missing_patchcore))
print("分割記録に無い PatchCore の行 / PatchCore rows not in manifest  :", len(unknown_patchcore))
print("較正用でない PatchCore の行 / PatchCore rows outside calibration:", len(wrong_partition))

incomplete = missing_yolo or missing_patchcore or unknown_patchcore or wrong_partition
if incomplete:
    print()
    for label, values in (("YOLO なし / no YOLO", missing_yolo),
                          ("PatchCore なし / no PatchCore", missing_patchcore),
                          ("分割記録に無い / not in manifest", unknown_patchcore),
                          ("較正用でない / not calibration", wrong_partition)):
        if values:
            print(f"{label}: {values[:5]}")
    if not ALLOW_PARTIAL_COMPARISON:
        raise ValueError(
            "較正用の画像がそろっていません。最終比較はできません / "
            "the calibration partition is incomplete; a final comparison is not possible")
    print()
    print("*** 探索用・非最終 / EXPLORATORY, NOT FINAL ***")
    print("足りない画像を外した部分集合で続けます。この結果を最終成績として報告しないでください。")
    print("Continuing with an excluded subset. Do not report this as a final result.")
    comparison_is_final = False
else:
    comparison_is_final = True

shared_images = manifest_calibration & patchcore_calibration & yolo_ids
excluded_no_yolo = missing_yolo
excluded_no_patchcore = missing_patchcore

if not shared_images:
    raise RuntimeError(
        "共通の画像が 1 枚もありません / there are no images with both results. "
        "第4日ノートブック03 と第5日ノートブック02 の出力先を確認してください。")

print()
print("両方そろった較正用画像 / calibration images with both results:", len(shared_images))
print("最終比較か / final comparison:", comparison_is_final)


#### 分割ごとに分ける / Split the images into their roles

| 分割<br>Partition | 何に使うか<br>What it is for | いつ見てよいか<br>When may you look? |
| --- | --- | --- |
| `patchcore_train` | PatchCore の学習（正常のみ）<br>PatchCore fitting (normal only) | 学習のときだけ<br>only while fitting <!-- taj-row: patchcore-train --> |
| `calibration` | **しきい値と判定規則を決める**<br>**choose the thresholds and the decision rule** | 何度でも見てよい<br>as often as you like <!-- taj-row: calibration --> |
| `test` | 最後に 1 回だけ成績を出す<br>the final score, once | **モデル・前処理・しきい値を固定してから**<br>**after the model, the preprocessing and the thresholds are frozen** <!-- taj-row: test --> |

見てよいかどうかは **「情報をどう使ったか」** で決まります。試験用データを見てからしきい値や
前処理を変えると、そのデータは「まだ見ていないデータ」ではなくなります。固定した設定で
同じ評価をもう一度実行するのは構いません（再現性の確認です）。

What matters is **how the information is used**. If you change a threshold or the preprocessing
after studying the test data, that data is no longer unseen. Repeating the same evaluation with the
frozen configuration is fine — that is a reproducibility check.

この研修の参考図は、**保存した 1 回の実行から作った公開済みの結果** です。参考図で試験用の例を
見てから設定を変えた場合、自分の手元の試験用データはもう未使用とは言えません。
新しい最終成績を出すには、**新しい未使用データ** が必要です。

The reference figures in this course are **published results of one recorded run**. If you change
settings after studying the test examples in those figures, your own test partition is no longer
untouched. **A new final result needs a fresh holdout.**

<!-- taj-unit: split-the-images-into-their-roles -->


In [ ]:
# ▸ 実行のみ / Run only：このまま実行します。書き換えません。
# ▸ Run only: press Shift + Enter and leave the code as it is.
def strict_side(frame, columns, label):
    subset = frame[frame["image"].isin(shared_images)].copy()
    if subset["image"].duplicated().any():
        raise ValueError(f"{label}: 同じ画像が複数行あります / duplicate image IDs")
    return subset.set_index("image")[columns]


yolo_side = strict_side(yolo_frame, ["partition", "truly_abnormal", "yolo_max_conf"], "YOLO")
patchcore_side = strict_side(patchcore_frame, ["partition", "truly_abnormal", "score"], "PatchCore")

combined = yolo_side.join(patchcore_side, rsuffix="_patchcore", how="inner")
combined = combined.rename(columns={"truly_abnormal": "truth_ng",
                                    "truly_abnormal_patchcore": "truth_ng_patchcore"})
mismatch = combined[combined["truth_ng"] != combined["truth_ng_patchcore"]]
if len(mismatch):
    print("正解ラベルが 2 つのファイルで食い違っています / ground truth disagrees between the two files:")
    print(mismatch[["truth_ng", "truth_ng_patchcore"]].head().to_string())
    raise ValueError("ground-truth mismatch / 正解ラベルの食い違い")

combined["truth_ng"] = combined["truth_ng"].astype(bool)

if set(combined["partition"]) != {"calibration"}:
    raise ValueError("較正用以外の行が混ざっています / non-calibration rows in the calibration frame")
if len(combined) != len(shared_images):
    raise ValueError("較正用の行数がそろっていません / the calibration rows do not pair up")

calibration_frame = combined.copy()
print("較正用 / calibration :", len(calibration_frame),
      "（正常 / normal:", int((~calibration_frame["truth_ng"]).sum()),
      ", 異常 / abnormal:", int(calibration_frame["truth_ng"].sum()), "）")
print("試験用のスコアはまだ読み込みません。固定後に封印予測を 1 回だけ使います。")
print("Test scores are not loaded yet; the sealed predictions are used once, after the freeze.")

# 学習画像が評価用に混ざっていないことを、ファイル名で再確認します。
# Re-check by file name: this also catches the same file stored in two folders.
train_names = {Path(name).name for name in manifest["partitions"]["patchcore_train"]}
test_names = {Path(name).name for name in manifest["partitions"]["test"]}
calibration_names = {Path(name).name for name in manifest["partitions"]["calibration"]}
leak = sorted((train_names & test_names) | (train_names & calibration_names))
assert not leak, f"学習画像が評価用に混ざっています / training images inside an evaluation partition: {leak[:3]}"
print()
print("学習画像と評価用のファイル名の重なり / training file names inside an evaluation partition:",
      len(leak))


## Part 4　同じ基準で OK/NG を出す / Making a common OK/NG decision

<!-- taj-unit: making-a-common-ok-ng-decision -->


#### しきい値は「較正用」で決める / Choose the thresholds on the calibration partition

前の版では、正常スコアの最大と異常スコアの最小からしきい値を決め、**同じデータで成績を出して** いました。これは「答えを見てから採点する」のと同じで、数字が良く出ます。やめます。

The earlier version derived a threshold from the maximum normal score and the minimum abnormal score **and then scored the same data** — the equivalent of marking your own exam. That is gone.

いまは 2 段階です。

Now there are two steps:

1. **較正用（calibration）** で 2 つのしきい値を決める。目的は 1 つ。 **正常品の誤報率（false reject rate）を 10% 以下に保ったまま、再現率（recall）を最大にする。** この 10% は **この研修の目標値** です。較正用の正常品は 6 枚しかないので、10% は 「誤報 0 枚」と同じ意味になります。**生産での誤報率を保証する数字ではありません。**<br>On **calibration**, choose both thresholds with one shared objective: **maximise recall subject to a false-reject rate of at most 10%.** The 10% figure is a **classroom objective**; with only six normal calibration images it means "zero observed false rejects". It is **not** a production-rate guarantee.
2. 決めた値を **固定** する。第4日ノートブック03 と第5日ノートブック02 が **試験用（TEST）** を採点・封印したあと、ここで両方を読み、比較表を作る。<br>**Freeze** them. Day04_03 and Day05_02 then score and seal **TEST**. Read both seals here to compute the comparison.

| 手法<br>Method | 判定規則<br>Decision rule | 決める値<br>Value chosen |
| --- | --- | --- |
| YOLO | `confidence ≥ C` の検出が 1 つ以上あれば NG<br>NG if at least one detection with `confidence >= C` | `C` <!-- taj-row: yolo --> |
| PatchCore | `score ≥ T` なら NG<br>NG if `score >= T` | `T` <!-- taj-row: patchcore --> |

2 つのしきい値は **そのままでは比べられません**。測っている量が違うからです
（YOLO は confidence、PatchCore は較正で正規化したスコアです）。たまたま同じ数値になることも
ありますが、それは「同じ意味」ではありません。比べるのは数値ではなく、
**同じ正解に対する画像単位の OK/NG の判定** です。

The two thresholds are **not directly comparable**: they measure different quantities (a detection
confidence and a score normalised on calibration). They could coincidentally hold the same number,
which would not make them the same thing. What is compared is **the image-level OK/NG decision
against the same ground truth**.

<!-- taj-unit: choose-the-thresholds-on-the-calibration-partition -->


### 較正用データの分布 / the distribution on calibration data

![](../../assets/reference_outputs/ref_score_distribution.png)

図：較正用（validation）データだけのスコア分布と、そこから決めたしきい値 0.50（保存値の 0〜1 スケール）。試験用データは含みません。

Figure: the score distribution on the calibration (validation) partition only, with the threshold 0.50 chosen there, on the saved 0–1 scale. The test partition is not included.

<!-- taj-unit: the-distribution-on-calibration-data-caption -->

**気づいてほしいこと / What to notice**

スコアは anomalib 2.7.0 の後処理で `clip((生の値 − しきい値) ÷ (較正の最大 − 最小) + 0.5, 0, 1)` に変換されており、**学習したしきい値が 0.5** になります。較正は正常 6 枚・異常 20 枚です。6 枚で誤報 0 件は「本番でも 0 件」の保証ではありません。

anomalib 2.7.0's post-processing maps the score with `clip((raw − threshold) / (cal_max − cal_min) + 0.5, 0, 1)`, so the **learned threshold becomes 0.5**. Calibration holds 6 normal and 20 abnormal images; zero false alarms in six does not guarantee zero in production.

<!-- taj-unit: what-to-notice -->

**考えてみてください / Try to answer**

- しきい値を 0.60 にすると、この較正データでは見逃しが何枚増えますか。<br>At a threshold of 0.60, how many misses appear on this calibration data?

<!-- taj-unit: try-to-answer -->


In [ ]:
# ▸ 実行のみ / Run only：このまま実行します。書き換えません。
# ▸ Run only: press Shift + Enter and leave the code as it is.
MAX_FALSE_REJECT_RATE = 0.10   # 正常品の誤報率の上限 / acceptable false-reject rate

# YOLO は 0.01 で推論してあるので、0.01 未満のしきい値は指定できません。
# YOLO was inferred at 0.01, so no threshold below 0.01 exists.
YOLO_MIN_CONF = float(yolo_data["min_inference_conf"])


def counts(predicted_ng, truth_ng):
    predicted_ng = pd.Series(predicted_ng).astype(bool)
    truth_ng = pd.Series(truth_ng).astype(bool)
    return {
        "TP": int((predicted_ng & truth_ng).sum()),
        "FP": int((predicted_ng & ~truth_ng).sum()),
        "FN": int((~predicted_ng & truth_ng).sum()),
        "TN": int((~predicted_ng & ~truth_ng).sum()),
    }


def choose_threshold(candidates, scores, truth, rule):
    """共通の目的関数で 1 つのしきい値を選ぶ / pick one threshold with the shared objective."""
    best = None
    for threshold in candidates:
        predicted = rule(scores, threshold)
        table = counts(predicted, truth)
        normals = table["FP"] + table["TN"]
        abnormal = table["TP"] + table["FN"]
        false_reject_rate = table["FP"] / normals if normals else float("nan")
        recall = table["TP"] / abnormal if abnormal else float("nan")
        if not normals or false_reject_rate > MAX_FALSE_REJECT_RATE:
            continue
        if not abnormal:
            continue
        key = (recall, threshold)
        if best is None or key > best["key"]:
            best = {"key": key, "threshold": float(threshold), "table": table,
                    "false_reject_rate": false_reject_rate, "recall": recall}
    return best


yolo_candidates = [YOLO_MIN_CONF + i * 0.01 for i in range(int((0.99 - YOLO_MIN_CONF) / 0.01) + 1)]
patchcore_candidates = [i / 100 for i in range(0, 101)]

yolo_best = choose_threshold(
    yolo_candidates,
    calibration_frame["yolo_max_conf"],
    calibration_frame["truth_ng"],
    lambda scores, threshold: scores >= threshold,
)
patchcore_best = choose_threshold(
    patchcore_candidates,
    calibration_frame["score"],
    calibration_frame["truth_ng"],
    lambda scores, threshold: scores >= threshold,
)

assert yolo_best is not None, "較正用データで目的を満たす YOLO しきい値がありません"
assert patchcore_best is not None, "較正用データで目的を満たす PatchCore しきい値がありません"

YOLO_THRESHOLD = yolo_best["threshold"]
PATCHCORE_THRESHOLD = patchcore_best["threshold"]

calibration_summary = pd.DataFrame([
    {"method": "YOLO", "threshold": round(YOLO_THRESHOLD, 3),
     "recall": round(yolo_best["recall"], 3),
     "false_reject_rate": round(yolo_best["false_reject_rate"], 3),
     **yolo_best["table"]},
    {"method": "PatchCore", "threshold": round(PATCHCORE_THRESHOLD, 3),
     "recall": round(patchcore_best["recall"], 3),
     "false_reject_rate": round(patchcore_best["false_reject_rate"], 3),
     **patchcore_best["table"]},
])
n_calibration_normal = int((~calibration_frame["truth_ng"]).sum())
max_allowed_fp = int(MAX_FALSE_REJECT_RATE * n_calibration_normal)
print(f"較正用の正常品は {n_calibration_normal} 枚なので、誤報率 "
      f"{MAX_FALSE_REJECT_RATE:.0%} は「誤報 {max_allowed_fp} 枚まで」という意味です。")
print(f"{n_calibration_normal} calibration normals -> the {MAX_FALSE_REJECT_RATE:.0%} cap allows "
      f"at most {max_allowed_fp} false alarm(s).")
print()
print("目的 / objective: recall を最大に、ただし正常品の誤報率 ≤ "
      f"{MAX_FALSE_REJECT_RATE:.0%} / maximise recall subject to false rejects ≤ "
      f"{MAX_FALSE_REJECT_RATE:.0%}")
print("（較正用データでの結果です / measured on the calibration partition）")
print()
print(calibration_summary.to_string(index=False))
print()
print("この 2 つの値を固定します / freezing these two values:")
print("  YOLO confidence threshold      :", round(YOLO_THRESHOLD, 3))
print("  PatchCore anomaly threshold    :", round(PATCHCORE_THRESHOLD, 3))

### 固定前の最終確認 / Last check before freezing

冒頭の主な流れに戻り、較正のみで選んだ2値とモデルmanifestを照合します。
試験を見て設定を変えたら、新しい最終評価には新しい未使用データが必要です。

Return to the primary flow and check both calibration-selected thresholds against the model manifest.
After tuning from test feedback, a new final assessment requires a fresh holdout.

<!-- taj-unit: last-check-before-freezing -->


#### しきい値を動かしてよいのは、較正用データだけ / You may move thresholds on the calibration data only

下のスライダーは **較正用データ** のスコア分布だけを使います。試験用のスコアは読み込んでいません。
**較正用の上で何度試しても、後の試験用の表は変わりません。**

The slider below uses **calibration** scores only; the test scores are not loaded into it.
**Trying as many values as you like on calibration does not change the later test table.**

ただし、**すでに試験用の結果を見てしまったあとでは**、較正用のスライダーを動かして選び直しても、
その判断は「試験用の結果を知った人」が下したものになります。**新しい最終成績には、新しい
未使用データが必要**です。

Once you have seen the test results, however, a choice made with the calibration slider is still a
choice made by someone who knows the test outcome. **A new final result needs a fresh holdout.**

誤報（FP）と見逃し（FN）は **いつも同じ方向に動くわけではありません**。
しきい値を上げると見逃しは増え、誤報は減ります。**同時に 0 にできる場合もあります** — 正常品の
スコアの最大値より、異常品のスコアの最小値が大きい（分布に隙間がある）ときです。
この研修で保存した実行では、較正用データにその隙間があったので、PatchCore は 0.50 で
**較正用の 26 枚（正常 6・異常 20）で** 見逃し 0・誤報 0 になりました。
**これは 1 回の記録された実行の値** で、どの環境でも同じにはなりません。
また、**しきい値を動かしても件数が変わらない区間** もあります（その間に入る画像が無いため）。
隙間が無いデータでは、どちらかを犠牲にするしかありません。

False alarms (FP) and misses (FN) do **not** always move in the same direction. Raising the
threshold adds misses and removes false alarms, and **both can reach zero at once** when the
distributions have a gap — the lowest abnormal score above the highest normal score. In the run
recorded for this course the calibration data had that gap, so PatchCore reached zero misses and
zero false alarms at 0.50 **on the 26 calibration images (6 normal, 20 abnormal)**. **That is one
recorded run**, not a result every machine reproduces. **Some threshold intervals change no counts at
all**, because no image falls between them. Without a gap you must sacrifice one side.

<!-- taj-unit: you-may-move-thresholds-on-the-calibration-data-only -->


### 較正で決めて、固定する / choose on calibration, then freeze

![](../../assets/diagrams/teach_threshold_freeze.png)

図：較正用データでしきい値を決めて固定する流れ（模式図）。図の YOLO 0.06、PatchCore 0.50 は過去の保存例です。今回の値は較正コードで決まります。

Figure: choosing a threshold on calibration data, then freezing it (schematic). The archived example froze YOLO at 0.06 and PatchCore at 0.50; live values come from calibration.

<!-- taj-unit: choose-on-calibration-then-freeze-caption -->

**気づいてほしいこと / What to notice**

固定した後に試験用データを見て動かすと、その数字はもう「未使用データでの成績」ではありません。較正用データは正常 6 枚・異常 20 枚なので、割合は 1/6 刻みでしか動きません。

If you move the threshold after seeing the test data, those numbers are no longer a result on untouched data. Calibration holds 6 normal images, so a false-reject rate can only move in steps of 1/6.

<!-- taj-unit: what-to-notice -->

**考えてみてください / Try to answer**

- 較正用の正常が 6 枚のとき、誤報率 10% 以下にするには誤報が何件以下である必要がありますか。<br>With six normal calibration images, how many false alarms are allowed for a false-reject rate of at most 10 %?

<!-- taj-unit: try-to-answer -->


In [ ]:
# ▸ この値を変更 / Change this value：指定の値だけを変え、結果を予想します。
# ▸ Change this value: edit only the marked value; predict before running.
# ウィジェットが無い環境のためのパラメータセル / parameter fallback: no widgets needed
import matplotlib.pyplot as plt

patchcore_threshold_to_try = 0.50   # ← この値を変える / change this value

predicted = calibration_frame["score"] >= patchcore_threshold_to_try
table = counts(predicted, calibration_frame["truth_ng"])
normals = table["FP"] + table["TN"]
abnormal = table["TP"] + table["FN"]
print(f"PatchCore しきい値 / threshold = {patchcore_threshold_to_try:.2f}")
print(f"較正用の誤報 FP = {table['FP']} / {normals}  見逃し FN = {table['FN']} / {abnormal}")
print("誤報率 / false-reject rate =",
      round(table["FP"] / normals, 3) if normals else "n/a")
print("再現率 / recall            =",
      round(table["TP"] / abnormal, 3) if abnormal else "n/a")

fig, ax = plt.subplots(figsize=(9.5, 3.6))
bins = [i / 50 for i in range(51)]
ax.hist(calibration_frame.loc[~calibration_frame["truth_ng"], "score"], bins=bins,
        color="#8fb6ff", edgecolor="#1f6feb", label="正常 / normal (calibration)")
ax.hist(calibration_frame.loc[calibration_frame["truth_ng"], "score"], bins=bins,
        color="#ffd580", edgecolor="#9a6700", label="異常 / abnormal (calibration)")
ax.axvline(patchcore_threshold_to_try, color="#a40e26", linewidth=2)
ax.set_xlabel("PatchCore 異常スコア / anomaly score（較正用データの正規化済み / normalised on calibration）")
ax.set_ylabel("枚数 / images")
ax.legend(fontsize=8)
ax.set_title(f"FP={table['FP']}  FN={table['FN']}（較正用 / calibration）", fontsize=9)
plt.tight_layout()
plt.show()


In [ ]:
# ▸ 実行のみ / Run only：このまま実行します。書き換えません。
# ▸ Run only: press Shift + Enter and leave the code as it is.
# スライダー版（較正用データのみ）。ウィジェットが無ければ下のパラメータセルを使ってください。
if not aivision.widgets_available():
    aivision.explain_widget_requirement()
else:
    import ipywidgets as widgets
    from IPython.display import clear_output, display

    output = widgets.Output()

    def draw(threshold):
        predicted = calibration_frame["score"] >= threshold
        table = counts(predicted, calibration_frame["truth_ng"])
        normals = table["FP"] + table["TN"]
        abnormal = table["TP"] + table["FN"]
        fig, ax = plt.subplots(figsize=(9.5, 3.6))
        bins = [i / 50 for i in range(51)]
        ax.hist(calibration_frame.loc[~calibration_frame["truth_ng"], "score"], bins=bins,
                color="#8fb6ff", edgecolor="#1f6feb", label="正常 / normal (calibration)")
        ax.hist(calibration_frame.loc[calibration_frame["truth_ng"], "score"], bins=bins,
                color="#ffd580", edgecolor="#9a6700", label="異常 / abnormal (calibration)")
        ax.axvline(threshold, color="#a40e26", linewidth=2)
        ax.set_xlabel("PatchCore 異常スコア / anomaly score（較正用 / calibration）")
        ax.set_ylabel("枚数 / images")
        ax.legend(fontsize=8)
        ax.set_title(f"threshold={threshold:.2f}   FP={table['FP']}  FN={table['FN']}", fontsize=9)
        plt.tight_layout()
        with output:
            clear_output(wait=True)
            plt.show()
            plt.close(fig)

    slider = widgets.FloatSlider(value=0.50, min=0.00, max=1.00, step=0.01,
                                 description="threshold",
                                 continuous_update=False)
    widgets.interact(draw, threshold=slider)
    display(output)

#### 必須：固定と未使用性を記録 / Required: record freeze and holdout status

較正の実験を終えてから、下の宣言を自分の言葉で記入します：
「試験を見る前に設定を固定し、試験を見た後は調整しません」。
モデル・前処理・しきい値の記録を確認してください。試験画像や結果を以前見た場合は
`TEST_ALREADY_SEEN=True` のままにします。その結果は**再利用データの説明用評価**です。
見ていない場合だけFalseにします。未記入では試験評価を実行できません。

**ステップ 3 の停止位置：** 固定記録を保存したら、次の封印読込セルはまだ実行しません。ステップ 4 と 5 を終えたら、ここに戻ります。

After calibration experiments, write your own declaration below:
“I froze settings before viewing test and will not tune after seeing test.”
Check the recorded model, preprocessing and thresholds. If test images/results were already seen,
keep `TEST_ALREADY_SEEN=True`: the result is an **illustrative reused-data assessment**.
Set False only if unseen. An empty declaration blocks test scoring.

**Stop here in step 3:** Save the freeze record, then leave the seal-reading cell unrun until steps 4 and 5 are complete.

<!-- taj-unit: required-record-freeze-and-holdout-status -->


較正スライダーは観察専用で、採用する2値を変更しません。目的に沿う値は較正選択セルを再実行して確認します。
この観察の後で固定し、固定記録の `assessment_role` を提出物に含めます。

The calibration slider is exploratory and does not change the two selected thresholds.
Re-run the calibration-selection cell to confirm values satisfying the declared objective.
Freeze only after exploration and include the record's `assessment_role` in your submission.

<!-- taj-unit: clarity-slider-before-freeze -->


In [ ]:
# ▸ この値を変更 / Change this value：指定の値だけを変え、結果を予想します。
# ▸ Change this value: edit only the marked value; predict before running.
FREEZE_DECLARATION = ""  # ← 自分の宣言を記入 / write your declaration
TEST_ALREADY_SEEN = True  # ← 未使用と確認できる場合だけFalse / False only for confirmed unseen test
freeze_path = COURSE_ROOT / "submissions" / "day05" / "comparison_freeze_record.json"
seal_paths = {
    "YOLO": COURSE_ROOT / "submissions" / "day04" / "yolo_predictions_sealed_test.json",
    "PatchCore": COURSE_ROOT / "submissions" / "day05" / "patchcore_test_predictions.json",
}
existing_seals = [label for label, path in seal_paths.items() if path.exists()]
if existing_seals and not freeze_path.is_file():
    globals().pop("freeze_record", None)
    raise RuntimeError(
        "封印があるのに元の固定記録がありません。新規作成せず、元の記録を確認してください / "
        "a test seal exists but its original freeze record is missing; cannot create a new freeze. "
        f"Keep the seals and recover the original record: {', '.join(existing_seals)}")
if existing_seals:
    freeze_record = json.loads(freeze_path.read_text(encoding="utf-8"))

if not globals().get("comparison_is_final"):
    missing = sorted(set(globals().get("missing_yolo") or [])
                     | set(globals().get("missing_patchcore") or []))
    raise RuntimeError(
        "較正用の画像がそろっていないため、固定できません。最終比較はできません / "
        "the calibration partition is incomplete, so nothing may be frozen and a final comparison is "
        "not possible.\n"
        "  足りない画像 / missing images: " + str(missing[:5]) + "\n"
        "  理由 / why: 較正用の画像が欠けたまま固定すると、2 つの手法を同じ画像で比べられません。/ "
        "freezing a partial calibration set means the two methods cannot be compared on the same "
        "images.\n"
        "  次にすること / next: 第4日ノートブック03 の Part 5 と第5日ノートブック02 の較正用セルを"
        "実行し直し、すべての較正用画像に 2 つの予測をそろえてください。/ re-run Part 5 of Day 4 "
        "Notebook 03 and the calibration cells of Notebook 02 until every calibration image has both "
        "predictions.")

if not FREEZE_DECLARATION.strip():
    raise ValueError("固定の宣言が必要 / write the freeze declaration before test")
candidate_freeze = {
    "manifest": manifest,
    "yolo_provenance": yolo_data,
    "declaration": FREEZE_DECLARATION.strip(),
    "test_already_seen": TEST_ALREADY_SEEN,
    "assessment_role": "illustrative_reused_data" if TEST_ALREADY_SEEN else "declared_untouched_test",
    "yolo_threshold": YOLO_THRESHOLD,
    "patchcore_threshold": PATCHCORE_THRESHOLD,
    "model": manifest["model"],
    "postprocessor": manifest["postprocessor"],
    "score_convention": manifest["score_convention"],
    "split": manifest["partitions"],
    "yolo_weights": {
        "path": yolo_data.get("weights"),
        "sha256": yolo_data.get("weights_sha256"),
    },
}
# 固定するのは「どの実験か」まで含めてです。名前ではなく、学習で決まった中身の指紋を入れます。
# Freezing covers *which* experiment: the fingerprint of the fitted bytes, not just a model name.
from experiment_identity import (ProvenanceError, canonical_json, check_file_sha256,
                                 frozen_identity, write_json_atomic)

fitted_state_path = COURSE_ROOT / "submissions" / "day05" / "patchcore_fitted_state.json"
if not fitted_state_path.is_file():
    raise RuntimeError(
        "学習の記録がありません。Day05_02_PatchCore_Training.ipynb の Part 3 の fit セルから"
        "実行し直してください / the fitted-state record is missing; re-run the fit cells of "
        "Day 5 Notebook 02 from Part 3")
fitted_record = json.loads(fitted_state_path.read_text(encoding="utf-8"))

calibrated_path = COURSE_ROOT / "submissions" / "day05" / "patchcore_calibrated_state.json"
if not calibrated_path.is_file():
    raise RuntimeError(
        "較正用を採点した記録がありません。第5日ノートブック02 の Part 4（較正用の採点）を"
        "実行し直してください / the post-calibration record is missing; re-run Part 4 of "
        "Day 5 Notebook 02 (scoring the calibration partition)")
calibrated_record = json.loads(calibrated_path.read_text(encoding="utf-8"))
base_experiment = calibrated_record["experiment"]

if (base_experiment["fitted"]["fingerprint"]
        != fitted_record["experiment"]["fitted"]["fingerprint"]):
    raise ValueError(
        "較正用の記録と学習の記録が同じモデルではありません / the calibration record and the "
        "fitted-state record describe different models; re-run Day 5 Notebook 02 from Part 3")
for key in ("schema", "dataset", "model", "score_convention"):
    if base_experiment.get(key) != (manifest.get("experiment") or {}).get(key):
        raise ValueError(
            "較正用の記録と分割記録が同じ実験ではありません / the calibration record and the split "
            "manifest describe different experiments; re-run Day 5 Notebook 02 from Part 3")
if patchcore_data.get("experiment") != base_experiment:
    raise ValueError(
        "較正用の予測が、この学習のものではありません / the calibration predictions come from "
        "another fit; re-run Day 5 Notebook 02 from Part 3")

candidate_freeze["experiment"] = frozen_identity(
    base_experiment,
    yolo_threshold=YOLO_THRESHOLD,
    patchcore_threshold=PATCHCORE_THRESHOLD,
)
candidate_freeze["model_fingerprint"] = base_experiment["fitted"]["fingerprint"]
candidate_freeze["experiment_id"] = candidate_freeze["experiment"]["experiment_id"]
candidate_freeze = json.loads(json.dumps(candidate_freeze))

# YOLO の重みは、パスだけでなく中身の指紋まで固定します。同じ名前で学習し直したファイルは、
# 別の実験です。
# Freeze the YOLO weights by content, not only by path: a retrained file under the same name is a
# different experiment.
frozen_weights = candidate_freeze["yolo_weights"]
if not frozen_weights.get("sha256"):
    raise ProvenanceError(
        "YOLO の重みの中身の指紋が較正用の予測にありません。第4日ノートブック03 の Part 5 を"
        "実行し直してください / the calibration predictions record no SHA-256 for the YOLO "
        "weights; re-run Part 5 of Day 4 Notebook 03")
if frozen_weights.get("path"):
    check_file_sha256(COURSE_ROOT / frozen_weights["path"], frozen_weights["sha256"],
                      what="YOLO の重み / the YOLO weights")

# 封印（試験用の採点）が 1 つでもできたあとは、固定記録は不変です。同じ設定のやり直しだけが同じ
# 記録になり、設定を変えるのは新しい実験になります。既存の封印結果は書き換えません。
# Once either test seal exists the freeze record is immutable. An identical rerun is a no-op; a
# changed setting must start a new experiment. Existing sealed results are never overwritten.
if existing_seals:
    changed = sorted(key for key in set(freeze_record) | set(candidate_freeze)
                     if freeze_record.get(key) != candidate_freeze.get(key))
    if canonical_json(freeze_record) == canonical_json(candidate_freeze):
        print("同じ固定記録です。書き換えません（封印済みの結果もそのまま残します）。")
        print("Identical freeze record: left untouched, and so are the existing sealed results.")
        print("封印 / seals:", ", ".join(existing_seals))
    else:
        raise RuntimeError(
            "封印済みの試験結果があるため、固定記録を書き換えられません。固定は不変です / "
            "the freeze record is immutable once a test seal exists.\n"
            f"  封印 / seals: {', '.join(existing_seals)}\n"
            f"  違う項目 / changed fields: {changed}\n"
            "  同じ設定でやり直したときだけ同じ記録になります。設定を変えるには、新しい実験として"
            "ステップ 1 からやり直してください（試験を一度見ているので TEST_ALREADY_SEEN=True の"
            "まま）。既存の封印ファイルは書き換えません。 / "
            "Only an identical rerun keeps this record. To change a setting, start a new experiment "
            "from step 1 with TEST_ALREADY_SEEN=True; the existing sealed files stay as they are.")
else:
    write_json_atomic(freeze_path, candidate_freeze)
    freeze_record = candidate_freeze
print("固定記録 / freeze record:", freeze_path)
print("評価の役割 / assessment role:", freeze_record["assessment_role"])
print("学習した状態の指紋 / fitted-state fingerprint:", freeze_record["model_fingerprint"][:16])
print("実験 ID / experiment ID:", freeze_record["experiment_id"][:16])


#### 封印した試験用予測を読み込む / Load the sealed test predictions

試験用のスコアは、**固定を記録したあと** に 2 つのノートブックが 1 回ずつ採点し、封印してあります。

- `Day04_03_YOLO_Evaluation.ipynb` の `shared-sealed-test`
  → `submissions/day04/yolo_predictions_sealed_test.json`
- `Day05_02_PatchCore_Training.ipynb` の `clarity-sealed-test-scoring`
  → `submissions/day05/patchcore_test_predictions.json`

このセルは 2 つの封印を読み、**実験の指紋・分割・モデル・しきい値・評価の役割が固定記録と一致する
こと** を確かめてから表にします。1 つでも一致しなければ停止し、封印ファイルはそのまま残します。
ここで採点し直すことはしません。試験用の 26 枚は、どちらの手法でも同じ画像でなければ比較できません。

The test scores were produced **after** the freeze, once per method, and sealed: by
`shared-sealed-test` in `Day04_03_YOLO_Evaluation.ipynb` (YOLO) and by
`clarity-sealed-test-scoring` in `Day05_02_PatchCore_Training.ipynb` (PatchCore). This cell reads both
seals and checks the **experiment fingerprint, the split, the model, the thresholds and the
assessment role** against the freeze record before building the table. It stops on any mismatch and
leaves the sealed files untouched; it never re-scores here. Both methods must have scored the same 26
test images, or there is nothing to compare.

<!-- taj-unit: load-the-sealed-test-predictions -->


In [ ]:
# ▸ 実行のみ / Run only：このまま実行します。書き換えません。
# ▸ Run only: press Shift + Enter and leave the code as it is.
# 固定後に、2 つの封印予測を読み込みます。どちらも固定記録と同じ実験のものでなければ使いません。
# After the freeze, read both seals; each must belong to the same frozen experiment.
import json

# A failed re-read must not leave the previous test frame available to the comparison cell.
test_frame = None
numeric_maps_status = None

from experiment_identity import (ProvenanceError, SEALED_SCHEMA, file_set_digest,
                                 normalise_image_id, test_partition, validate_sealed,
                                 validate_yolo_sealed)

expected = freeze_record["experiment"]
expected_role = freeze_record.get("assessment_role")

patchcore_sealed_path = COURSE_ROOT / "submissions" / "day05" / "patchcore_test_predictions.json"
yolo_sealed_path = COURSE_ROOT / "submissions" / "day04" / "yolo_predictions_sealed_test.json"

for label, path in (("PatchCore", patchcore_sealed_path), ("YOLO", yolo_sealed_path)):
    if not path.is_file():
        raise RuntimeError(
            f"{label} の封印テスト予測がありません / no sealed {label} test predictions:\n  {path}\n"
            "固定を記録したあとで、次の 2 つを 1 回ずつ実行してください / after the freeze, run each "
            "of these once:\n"
            "  4. Day04_03_YOLO_Evaluation.ipynb   → shared-sealed-test\n"
            "  5. Day05_02_PatchCore_Training.ipynb → clarity-sealed-test-scoring")

patchcore_sealed = json.loads(patchcore_sealed_path.read_text(encoding="utf-8"))
yolo_sealed = json.loads(yolo_sealed_path.read_text(encoding="utf-8"))

try:
    # 形式・実験の指紋・分割・しきい値・評価の役割・画像の網羅・重複・型をまとめて確かめます。
    # One call per method checks the schema, the experiment fingerprint, the split, the threshold,
    # the assessment role, the image coverage, the duplicate IDs and the value types.
    patchcore_rows = validate_sealed(patchcore_sealed, schema_version=SEALED_SCHEMA,
                                     expected=expected, manifest=manifest,
                                     expected_role=expected_role)
    yolo_rows = validate_yolo_sealed(yolo_sealed, schema_version=1, expected=expected,
                                     manifest=manifest, expected_role=expected_role)
except ProvenanceError as error:
    print("封印予測を採用できません / a sealed file cannot be used:")
    print(str(error))
    print()
    print("封印ファイルはそのまま残しています。採点し直しません。")
    print("The sealed files are untouched and test is not re-scored.")
    print("封印や固定記録を削除せず、講師と実験 ID と保存先を確認してください。")
    print("Keep the seals and freeze record; check the experiment ID and paths with your instructor.")
    raise

sealed_frame = pd.DataFrame(patchcore_rows)
sealed_frame["image"] = sealed_frame["image"].map(normalise_image_id)
sealed_frame = sealed_frame.set_index("image")

yolo_test_frame = pd.DataFrame(yolo_rows)
yolo_test_frame["image"] = yolo_test_frame["image"].map(normalise_image_id)
yolo_test_frame = yolo_test_frame.set_index("image")

if sealed_frame.index.duplicated().any() or yolo_test_frame.index.duplicated().any():
    raise ValueError("封印予測に同じ画像が複数行あります / duplicate image IDs in a sealed file")
if set(sealed_frame.index) != set(yolo_test_frame.index):
    raise ValueError(
        "YOLO と PatchCore の試験用の画像が違います。2 つの手法は同じ画像で採点しなければ"
        "比較できません / the two methods were not scored on the same test images: "
        f"only PatchCore={sorted(set(sealed_frame.index) - set(yolo_test_frame.index))[:5]} "
        f"only YOLO={sorted(set(yolo_test_frame.index) - set(sealed_frame.index))[:5]}")
if not set(sealed_frame.index) <= test_partition(manifest):
    raise ValueError("試験用でない画像があります / sealed rows outside the test partition")

if bool((sealed_frame["truly_abnormal"].astype(bool)
         != yolo_test_frame["truly_abnormal"].astype(bool)).any()):
    raise ValueError("正解ラベルが 2 つのファイルで食い違っています / ground truth disagrees")

# 数値マップは可視化用の副産物で、正本は JSON のスコアです。それでも、古いマップや欠けたマップを
# 今回の実験の成果として見せないよう、封印に記録された指紋と比べます。
# The numeric maps are visualisations; the JSON scores are authoritative. The recorded digest is
# still checked, so a stale or incomplete map set is never presented as this experiment's output.
numeric_maps_status = {"recorded": patchcore_sealed.get("maps"), "verified": False}
maps_record = patchcore_sealed.get("maps")
if isinstance(maps_record, dict) and maps_record.get("directory") == "maps":
    map_dir = patchcore_sealed_path.parent / "maps"
    published_maps = sorted(map_dir.glob("*_map.npz")) if map_dir.is_dir() else []
    expected_names = {f"{Path(row['image']).stem}_map.npz" for row in patchcore_rows}
    if len(expected_names) == len(patchcore_rows) and {p.name for p in published_maps} == expected_names:
        try:
            actual_maps = file_set_digest(published_maps)
        except OSError:
            pass  # An unreadable optional map cannot be evidence; the sealed scores still can.
        else:
            numeric_maps_status["actual"] = actual_maps
            numeric_maps_status["verified"] = (
                actual_maps["count"] == maps_record.get("count")
                and actual_maps["digest"] == maps_record.get("digest"))
if not numeric_maps_status["verified"]:
    print("注意 / caution: 数値マップを確認できません。今回の成果として表示せず、JSON のスコアを使います。")
    print("The numeric maps do not match the sealed record or could not be verified. Use the JSON scores.")

test_frame = pd.DataFrame({
    "partition": "test",
    "truth_ng": sealed_frame["truly_abnormal"].astype(bool),
    "yolo_max_conf": yolo_test_frame["yolo_max_conf"].astype(float),
    "score": sealed_frame["score"].astype(float),
})

print("Existing sealed predictions match this frozen experiment.")
print("2 つの封印予測を読み込みました / both seals loaded")
print("実験 ID / experiment ID:", expected["experiment_id"][:16])
print("評価の役割 / assessment role:", expected_role)
print("採点日時 / scored at (UTC): PatchCore", patchcore_sealed.get("scored_at_utc"),
      "/ YOLO", yolo_sealed.get("scored_at_utc"))
print("試験用 / test :", len(test_frame),
      "（正常 / normal:", int((~test_frame["truth_ng"]).sum()),
      ", 異常 / abnormal:", int(test_frame["truth_ng"].sum()), "）")
print("固定の宣言 / freeze declaration:", freeze_record["declaration"][:60], "...")


#### 固定したしきい値で、試験用を 1 回だけ評価する / Score the test partition once, with the frozen thresholds

しきい値は較正用データで決めて、**ここで固定** しました。これから先、試験用の数字を見て
しきい値を動かすと、その表は「答えを見てから採点した表」になります。
動かしたくなったら、前の較正用スライダーを **較正用データ** で使ってください。
試験を既に見た場合、変更した設定には **別の未使用データ** が必要です。同じ試験での再評価を、元の最終試験成績として示してはいけません。

The thresholds were chosen on calibration and are **frozen here**. Tuning them after seeing these
numbers turns this table into marking your own exam. If you want to move one, use the earlier calibration slider
on **calibration** data. If test was already inspected, a revised configuration needs a **new untouched holdout**;
do not present reassessment on the same test as the original final test score.

**重み・前処理・正常メモリ・後処理も固定**します。試験を見てこれらを変えてはいけません。
同梱図の 0.06 / 0.50 は保存済み参考実行の例で、今回の実行の期待値ではありません。

**Freeze weights, preprocessing, normal memory and postprocessing too.** Do not change them
after inspecting test results. Supplied 0.06 / 0.50 values are saved reference-run examples,
not expected results for your live run.

<!-- taj-unit: score-the-test-partition-once-with-the-frozen-thresholds -->


In [ ]:
# ▸ 実行のみ / Run only：このまま実行します。書き換えません。
# ▸ Run only: press Shift + Enter and leave the code as it is.
import json
from datetime import datetime, timezone

if "freeze_record" not in globals() or not freeze_record["declaration"]:
    raise RuntimeError("先に固定宣言を記録 / record the freeze declaration first")
if globals().get("test_frame") is None:
    raise RuntimeError("先に 2 つの封印を読み込む / load and verify both seals first")
if not globals().get("comparison_is_final"):
    raise RuntimeError(
        "較正用の画像がそろっていないため、最終評価はできません。探索用の結果を最終成績として"
        "保存しません / the calibration partition is incomplete, so no final evaluation is possible; "
        "an exploratory result is never saved as a final one")
if (manifest != freeze_record["manifest"] or yolo_data != freeze_record["yolo_provenance"]
        or YOLO_THRESHOLD != freeze_record["yolo_threshold"]
        or PATCHCORE_THRESHOLD != freeze_record["patchcore_threshold"]
        or manifest["model"] != freeze_record["model"]
        or manifest["postprocessor"] != freeze_record["postprocessor"]):
    raise RuntimeError("固定後に設定が変わりました / frozen settings changed; do not reuse test")
frozen_weights = freeze_record.get("yolo_weights") or {}
if not frozen_weights.get("sha256"):
    raise RuntimeError(
        "固定記録に YOLO の重みの指紋がありません。第5日ノートブック03 の固定からやり直してください / "
        "the freeze record carries no YOLO weight digest; redo the freeze in Day 5 Notebook 03")
if frozen_weights.get("sha256") != yolo_data.get("weights_sha256"):
    raise RuntimeError(
        "固定記録の YOLO の重みと、較正用の予測の重みが違います / the frozen YOLO weights differ "
        "from the weights behind the calibration scores; redo the freeze")
test_predictions = {
    "YOLO": test_frame["yolo_max_conf"] >= YOLO_THRESHOLD,
    "PatchCore": test_frame["score"] >= PATCHCORE_THRESHOLD,
}
truth = test_frame["truth_ng"]

rows = []
for method, predicted in test_predictions.items():
    table = counts(predicted, truth)
    tpr = table["TP"] / (table["TP"] + table["FN"]) if (table["TP"] + table["FN"]) else float("nan")
    fpr = table["FP"] / (table["FP"] + table["TN"]) if (table["FP"] + table["TN"]) else float("nan")
    precision = table["TP"] / (table["TP"] + table["FP"]) if (table["TP"] + table["FP"]) else float("nan")
    rows.append({
        "method": method,
        "threshold": round(YOLO_THRESHOLD if method == "YOLO" else PATCHCORE_THRESHOLD, 3),
        **table,
        "n_images": table["TP"] + table["FP"] + table["FN"] + table["TN"],
        "n_abnormal": table["TP"] + table["FN"],
        "n_normal": table["FP"] + table["TN"],
        "recall": round(tpr, 3) if tpr == tpr else float("nan"),
        "false_reject_rate": round(fpr, 3) if fpr == fpr else float("nan"),
        "precision": round(precision, 3) if precision == precision else float("nan"),
    })

comparison = pd.DataFrame(rows)
# 評価の役割を結果そのものに残します。CSV だけを見ても、何の評価か分かるようにします。
# Carry the assessment role into the result itself, so a CSV alone still says what it is.
comparison.insert(0, "assessment_role", freeze_record["assessment_role"])
print("試験用・固定済み / frozen TEST:", freeze_record["assessment_role"])
print("分母 / denominators: 異常 n=", int((truth).sum()), ", 正常 n=", int((~truth).sum()))
print()
print(comparison.to_string(index=False))
print()
if comparison["precision"].isna().any() or comparison["recall"].isna().any():
    print("注意 / caution: nan は「0 件で計算できない」という意味です。0 として扱わないでください。")
    print("nan means the denominator was empty. It does not mean zero.")

# 欠陥の種類ごとの内訳（件数が少ないので、参考として読むこと）
per_type = []
defect_types = test_frame.index.str.split("/").str[0]
for method, predicted in test_predictions.items():
    for name in sorted(set(defect_types)):
        mask = defect_types == name
        table = counts(predicted[mask], truth[mask])
        per_type.append({"method": method, "defect_type": name, **table})
per_type_frame = pd.DataFrame(per_type)
print()
print("欠陥の種類ごと（試験用）/ per defect type on the test partition")
print(per_type_frame.to_string(index=False))

# 古い比較結果は「この方法で出した結果」ではないので、上書きせず別名で保存します。
# The old comparison was produced by a different method: never overwrite it.
results = {
    "schema_version": 3,
    "assessment_role": freeze_record["assessment_role"],
    "assessment_is_final": bool(comparison_is_final),
    "freeze_record": freeze_record,
    "generated_at_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "supersedes": {
        "file": "yolo_vs_patchcore.csv",
        "status": "superseded",
        "reason": (
            "The earlier comparison chose the PatchCore threshold from the maximum normal and "
            "minimum abnormal score of the same merged dataset it then evaluated. It is kept as "
            "historical evidence and must not be read as a result of the current method."
        ),
    },
    "split_manifest": str(manifest_path.relative_to(COURSE_ROOT)),
    "objective": {
        "function": "maximise recall subject to false_reject_rate <= MAX_FALSE_REJECT_RATE",
        "max_false_reject_rate": MAX_FALSE_REJECT_RATE,
        "selected_on": "calibration partition only",
    },
    "calibration": calibration_summary.to_dict(orient="records"),
    "test": comparison.to_dict(orient="records"),
    "per_defect_type": per_type_frame.to_dict(orient="records"),
    "score_convention": manifest["score_convention"],
    "model_identity": manifest["model"],
    "yolo_weights": freeze_record.get("yolo_weights"),
    "yolo_min_inference_conf": YOLO_MIN_CONF,
    "decisions": {
        "YOLO": "NG if at least one detection with conf >= threshold",
        "PatchCore": "NG if image anomaly score >= threshold",
    },
    "coverage": {
        "images_with_both_results": len(shared_images),
        "excluded_no_yolo": missing_yolo,
        "excluded_no_patchcore": missing_patchcore,
    },
    "numeric_maps": globals().get("numeric_maps_status"),
}

results_path = COURSE_ROOT / "submissions" / "day05" / "comparison_test_results.json"
from experiment_identity import write_json_atomic

write_json_atomic(results_path, results)

# CSV も置き換えで公開します。途中で止まった半端な CSV を「最終結果」として読めないようにします。
# Publish the CSV by replacement too: an interrupted write must not leave a partial file that reads
# like a final result. The BOM keeps Excel opening it as UTF-8.
from experiment_identity import write_text_atomic

csv_path = COURSE_ROOT / "submissions" / "day05" / "comparison_test_results.csv"
write_text_atomic(csv_path, "\ufeff" + comparison.to_csv(index=False))
print()
print("保存 / saved:", results_path)
print("保存 / saved:", csv_path)

### 実行結果の例 / Expected output

この結果は、この研修で保存した実行の記録です（モデル・データ分割・seed は本文に記載）。
細かな数値は環境で変わりますが、傾向と大小関係を読み取ってください。`x` のままの箇所は、
あなたの環境で決まる値です。

```text
分割の件数 / partition sizes: {'patchcore_train': 32, 'calibration': 26, 'test': 26}
分割の重なり / overlap check  : {'patchcore_train & calibration': 0, 'patchcore_train & test': 0, 'calibration & test': 0}
分割が決まらない行 / rows without a partition: 0 0

較正用 / calibration : 26 （正常 / normal: 6 , 異常 / abnormal: 20 ）
試験用 / test        : 26 （正常 / normal: 6 , 異常 / abnormal: 20 ）

較正用の正常品は 6 枚なので、誤報率 10% は「誤報 0 枚まで」という意味です。
目的 / objective: recall を最大に、ただし正常品の誤報率 ≤ 10%

method     threshold  recall  false_reject_rate  TP  FP  FN  TN
YOLO           0.xxx   x.xxx              x.xxx   x   x   x   x
PatchCore      0.xxx   x.xxx              x.xxx   x   x   x   x

試験用データでの結果（しきい値は固定済み）/ results on the untouched test partition
分母 / denominators: 異常 n= 20 , 正常 n= 6
```

These numbers come from the run recorded for this course (model, split and seed are stated in the
text). Exact values vary between machines; read the trends and the ordering. Where an `x` remains,
your own run decides the value.

The recorded run froze the two thresholds below. Read the first block as **calibration** (where the
thresholds were chosen) and the second as the **one test evaluation**:

```text
calibration (26 images, 6 normal / 20 abnormal)   threshold  recall  false_reject_rate  TP FP FN TN
YOLO                                                  0.060     1.00                0.00  20  0  0  6
PatchCore                                             0.500     1.00                0.00  20  0  0  6

test (26 images, 6 normal / 20 abnormal)          threshold  recall  false_reject_rate  TP FP FN TN
YOLO                                                  0.060     1.00                0.00  20  0  0  6
PatchCore                                             0.500     0.95                0.00  19  0  1  6
```

<!-- taj-unit: expected-output -->


## Part 5　結果を並べて見る / Looking at them side by side

<!-- taj-unit: looking-at-them-side-by-side -->


### 比較の前に：3パネルの読み方 / Before comparing: read three heatmap panels

![同じ合成画像の元画像・マップ・重ね表示 / synthetic original, map and overlay](../../assets/diagrams/teach_score_map_overlay.png)

<!-- taj-unit: before-comparing-read-three-heatmap-panels -->

**気づいてほしいこと / What to notice**

この模式図は **3パネル**：元画像・異常マップ・重ね表示です。実際のモデル出力でも5列比較でもありません。
**考えてみてください / Try to answer**

予想：色の範囲だけを変えたらスコアは変わりますか。カラーバーで場所の手がかりを読みます。

This schematic has **three panels**: original, anomaly map and overlay. It is neither actual model
output nor a five-column comparison. Predict: does changing only the colour range change the score?
Read the colourbar to interpret location clues.

<!-- taj-unit: what-to-notice -->


### 固定後の付録：保存済みTESTの5列 / After-freeze appendix: archived TEST in five columns

次の任意セルだけが保存済みTESTの図を表示します。最初の未使用試験を終えるまでFalseを保ちます。
表示後は、このデータで再選択した設定を「未使用試験の成績」と呼べません。

Only the optional cell below displays the archived TEST panel. Leave it False until your first
untouched assessment is complete. After viewing it, retuned settings cannot claim an untouched-test result.

<!-- taj-unit: after-freeze-appendix-archived-test-in-five-columns -->


In [ ]:
# ▸ この値を変更 / Change this value：指定の値だけを変え、結果を予想します。
# ▸ Change this value: edit only the marked value; predict before running.
SHOW_ARCHIVED_TEST = False  # ← 最終評価後だけTrue / True only after final assessment
if SHOW_ARCHIVED_TEST:
    if "comparison" not in globals() or "freeze_record" not in globals():
        raise RuntimeError("先に固定して評価 / freeze and evaluate before opening archive")
    from IPython.display import Markdown, display
    from experiment_identity import write_json_atomic
    print("過去のTEST出力・今回の実行ではない / ARCHIVED TEST, not this run")
    archive_path = COURSE_ROOT / "docs" / "instructor" / "day05_after_freeze_archive.md"
    display(Markdown(archive_path.read_text(encoding="utf-8")))

    # 封印予測が既にあるなら、固定記録の役割を後から書き換えません。書き換えると「封印した
    # 試験」と「固定記録」の役割が食い違い、どちらが本当か証明できなくなります。
    # Never rewrite the role after test is sealed: the seal and the record would then disagree, and
    # nothing could prove which role is true.
    sealed_exists = any((COURSE_ROOT / "submissions" / path).exists() for path in (
        "day04/yolo_predictions_sealed_test.json", "day05/patchcore_test_predictions.json"))
    if sealed_exists:
        print("封印予測が既にあります。固定記録の評価の役割は書き換えません。")
        print("A seal already exists, so the freeze record's assessment role is NOT rewritten.")
        print("同じ画像をもう一度使うときは、新しい実験として TEST_ALREADY_SEEN=True から始めてください。")
        print("To use these images again, start a new experiment with TEST_ALREADY_SEEN=True.")
    else:
        freeze_record["test_already_seen"] = True
        freeze_record["assessment_role"] = "illustrative_reused_data"
        freeze_record["assessment_role_downgraded"] = (
            "archived TEST panels were opened before test was sealed; the role can only improve to "
            "declared_untouched_test by starting a new experiment")
        write_json_atomic(freeze_path, freeze_record)
        print("評価の役割を illustrative_reused_data に下げて記録しました（改善はできません）。")
        print("Recorded a downgrade to illustrative_reused_data; it can never be upgraded here.")
else:
    print("TEST保存図は非表示 / archived TEST panel hidden")


### ここからはCALIBRATION / CALIBRATION gallery starts here

次のギャラリーは較正画像で、上の保存済みTESTの3画像とは別です。
ディスク上のフォルダ名 `test/` ではなく、分割manifestが役割を決めます。

The next gallery uses calibration images, not the three archived TEST images above.
Split-manifest membership, not the on-disk folder name `test/`, determines the role.

<!-- taj-unit: calibration-gallery-starts-here -->


In [ ]:
# ▸ 実行のみ / Run only：このまま実行します。書き換えません。
# ▸ Run only: press Shift + Enter and leave the code as it is.
# ここに並ぶのは第5日ノートブック02 が保存した **較正用データ** の重ね表示です。
# 表示範囲は 0〜1 に固定して保存してあります / the saved overlays use the fixed 0-1 range.
# These overlays were saved from CALIBRATION images by Day 5 Notebook 02.
heatmap_dir = COURSE_ROOT / "submissions" / "day05" / "heatmaps"
heatmap_files = sorted(heatmap_dir.glob("*_heatmap.png")) if heatmap_dir.is_dir() else []

print("CALIBRATION / 較正用 — not TEST:", len(heatmap_files))
for path in heatmap_files:
    print("  ", path.name)

if heatmap_files:
    show([Image.open(path) for path in heatmap_files],
         ["CALIBRATION: " + path.stem.replace("_heatmap", "") for path in heatmap_files],
         figsize=(4.2 * len(heatmap_files), 4.2))
else:
    print()
    print("ヒートマップがありません。第5日ノートブック02 を実行してください。")
    print("No heatmaps; run Day 5 Notebook 02.")


In [ ]:
# ▸ 実行のみ / Run only：このまま実行します。書き換えません。
# ▸ Run only: press Shift + Enter and leave the code as it is.
# 元の画像と並べる / compare with the original images
# 較正用の画像も test/ の下にあるので、同じ検索で元画像が見つかります。
# Calibration images also live under test/, so the same search finds the originals.
test_root = ASSETS / "anomaly_dataset" / "pvc_sheet" / "test"
if heatmap_files:
    pairs = []
    titles = []
    for path in heatmap_files[:4]:
        stem = path.stem.replace("_heatmap", "")
        original = next(test_root.rglob(f"{stem}.png"), None)
        if original is None:
            continue
        pairs.append(Image.open(original))
        titles.append(f"CALIBRATION original: {stem}")
    if pairs:
        show(pairs, titles, figsize=(4.2 * len(pairs), 4.2))


#### 比較の解釈 / Interpreting the comparison

ヒートマップは場所、YOLOは学習したクラスの手がかりです。組合せの効果は保証されません。
速度と誤りを測って判断します。保存済みTESTの具体的な失敗例は、固定後の任意付録だけにあります。
合成と実物の差、少数データ、ラベル予算、解像度・前処理・しきい値への依存を報告します。

A heatmap suggests location; YOLO suggests learned classes. Combining them is not guaranteed to help:
measure speed and errors. Specific archived TEST failures appear only in the optional after-freeze appendix.
Report synthetic-to-real limits, small samples, label budgets, and dependence on resolution,
preprocessing and thresholds.

<!-- taj-unit: interpreting-the-comparison -->


---


## 演習 / Exercises

<!-- taj-unit: exercises -->


#### 演習 1（基本）：比較表を完成させる / Exercise 1 (basic): complete the comparison table

下の表を、あなたの結果で埋めてください。

| 項目<br>Item | YOLO | PatchCore |
| --- | --- | --- |
| 学習に必要な画像<br>images needed for fitting | | <!-- taj-row: images-needed-for-fitting --> |
| 学習に必要なラベル<br>labels needed for fitting | | <!-- taj-row: labels-needed-for-fitting --> |
| 出力<br>output | | <!-- taj-row: output --> |
| 欠陥の種類が分かるか<br>names the defect type | | <!-- taj-row: names-the-defect-type --> |
| 位置が分かるか<br>shows the location | | <!-- taj-row: shows-the-location --> |
| 見逃し（FN）の数<br>number of misses (FN) | | <!-- taj-row: misses --> |
| 誤報（FP）の数<br>number of false alarms (FP) | | <!-- taj-row: false-alarms --> |

Fill in the table with your own results.
Done when every cell is filled with a concrete answer or a number from your run.

<!-- taj-unit: exercise-1-basic-complete-the-comparison-table -->


In [ ]:
# ▸ 自分でコードを書く / Write your code：空欄を自分で埋めてから実行します。
# ▸ Write your code: fill in the blanks, then run.
# 演習 1 / Exercise 1
rows = [
    ("学習に必要な画像 / training images", "", ""),
    ("学習に必要なラベル / labels", "", ""),
    ("出力 / output", "", ""),
    ("欠陥の種類が分かるか / defect type", "", ""),
    ("位置が分かるか / location", "", ""),
    ("見逃し FN / misses", "", ""),
    ("誤報 FP / false alarms", "", ""),
]
for item, yolo_answer, patchcore_answer in rows:
    print(f"{item:<38} YOLO={yolo_answer or '-':<12} PatchCore={patchcore_answer or '-'}")

#### 演習 2（中級）：較正用データでしきい値を動かす / Exercise 2 (intermediate): move the threshold on the calibration data

PatchCore のしきい値を 0.30 / 0.50 / 0.80 に変えて、**見逃し（FN）と誤報（FP）** が
どう変わるかを **3 行の表** にしてください。**しきい値の変更には較正用データだけを使います。**
試験用データで同じことをすると、その表は「答えを見てから採点した表」になります。

| しきい値<br>Threshold | 見逃し FN（分母: 異常 20）<br>Misses FN (of 20 abnormal) | 誤報 FP（分母: 正常 6）<br>False alarms FP (of 6 normal) |
| --- | --- | --- |
| 0.30 | | |
| 0.50 | | |
| 0.80 | | |

**できたら / Done when:** 3 行の表ができ、FN と FP が逆向きに動く（同時には動かないこともある）
ことを一文で書いた。件数が変わらない行があれば、その理由も書いてください。

Vary the PatchCore threshold over 0.30 / 0.50 / 0.80 and record how the number of misses (FN) and
false alarms (FP) changes in a **three-row table**. **Change the threshold using calibration data
only.** Doing the same on the test partition would turn that table into marking your own exam.

**Done when:** the three-row table is complete and one sentence states that FN and FP move in
opposite directions (not necessarily at the same time). If a row's counts do not change, say why.

<!-- taj-unit: exercise-2-intermediate-move-the-threshold-on-the-calibration-data -->


In [ ]:
# ▸ この値を変更 / Change this value：値を変えて、結果を予想してから実行します。
# ▸ Change this value: edit it below, predict the result, then run.
# 演習 2 / Exercise 2
# 較正用データでだけ動かします。試験用データは汚しません。
# Explore on the CALIBRATION partition only; the test partition stays untouched.
thresholds_to_try = [0.30, 0.50, 0.80]      # ← PatchCore のしきい値 / anomaly thresholds

if len(calibration_frame):
    for threshold in thresholds_to_try:
        predicted = calibration_frame["score"] >= threshold
        truth = calibration_frame["truth_ng"]
        fn = int((~predicted & truth).sum())
        fp = int((predicted & ~truth).sum())
        print(f"calibration threshold={threshold:.2f}  FN={fn}  FP={fp}")
    print()
    print("試験用データの結果は変わりません。しきい値を決め直したら、新しい試験データが必要です。")
    print("The test results do not change. Re-tuning a threshold means you need fresh test data.")
else:
    print("較正用データがありません / no calibration data")

#### 演習 3（発展・任意）：塩ビ検査への推奨を作る / Exercise 3 (optional challenge): build a recommendation for PVC

次の構成で、**あなたの推奨** を書いてください。根拠として、
この 5 日間で得た数字や観察を必ず引用してください。

| 項目<br>Item | あなたの推奨<br>Your recommendation |
| --- | --- |
| 第 1 段階で使う手法<br>method for the first stage | <!-- taj-row: first-stage-method --> |
| その理由<br>the reason | <!-- taj-row: the-reason --> |
| 第 2 段階で追加する手法<br>method added in the second stage | <!-- taj-row: second-stage-method --> |
| 見逃しを減らすためにすること<br>how to reduce misses | <!-- taj-row: reduce-misses --> |
| 誤報を減らすためにすること<br>how to reduce false alarms | <!-- taj-row: reduce-false-alarms --> |
| 第 2 週で確認すべきこと<br>what to confirm in week 2 | <!-- taj-row: week-2-checks --> |

Write your recommendation in the table above, citing numbers or observations from the
past five days as evidence.
Done when every row is filled and at least two rows quote a concrete result from this course.

<!-- taj-unit: exercise-3-optional-challenge-build-a-recommendation-for-pvc -->


In [ ]:
# ▸ 自分でコードを書く / Write your code：空欄を自分で埋めてから実行します。
# ▸ Write your code: fill in the blanks, then run.
# 演習 3 / Exercise 3
recommendation = {
    "stage1_method": "",
    "reason": "",
    "stage2_method": "",
    "reduce_misses": "",
    "reduce_false_alarms": "",
    "week2_checks": "",
}
for key, value in recommendation.items():
    print(f"{key:<22} {value or '(未記入 / not written)'}")

---


#### 理解度チェック / Understanding questions

1. YOLO の mAP と PatchCore の AUROC を直接比べてはいけないのはなぜですか。<br>Why must YOLO's mAP and PatchCore's AUROC not be compared directly?
2. 画像単位で比べるために、両方に何が必要ですか。<br>What does each method need in order to be compared at image level?
3. PatchCore が見逃しやすいのは、どんな欠陥ですか。<br>What kind of defect is PatchCore likely to miss?
4. YOLO が見逃しやすいのは、どんな欠陥ですか。<br>What kind of defect is YOLO likely to miss?
5. あなたの現場なら、どちらを先に試しますか。理由も書いてください。<br>On your line, which would you try first, and why?

Model answers: `docs/instructor/day05_instructor_guide.md`.

<!-- taj-unit: understanding-questions -->


---


## 困ったときは / Troubleshooting

エラーが出たら、まず「どのセルで」「どの行で」「どんなメッセージが出たか」を 書き留めてください。提出物にも記録します。

When an error appears, first write down *which cell*, *which line*, and *what the message says*. You will also record it in your submission.

| 症状<br>Symptom | 対処<br>What to do |
| --- | --- |
| `yolo_predictions_on_shared_set.json` が無い<br>`yolo_predictions_on_shared_set.json` is missing | 第4日ノートブック03 の Part 5 を実行してください。<br>Run Part 5 of Day 4 Notebook 03. <!-- taj-row: yolo-predictions-missing --> |
| `patchcore_predictions.json` が無い<br>`patchcore_predictions.json` is missing | 第5日ノートブック02 の Part 5 を実行してください。<br>Run Part 5 of Day 5 Notebook 02. <!-- taj-row: patchcore-predictions-missing --> |
| 共通の画像が 0 枚になる<br>Zero common images | パスの形式が違う可能性があります。`image` 列の先頭 `test/` を外しているか確認してください。<br>Path formats may differ. Check that the leading `test/` was stripped. <!-- taj-row: zero-common-images --> |
| precision や recall が `nan` になる<br>precision/recall are `nan` | TP + FP または TP + FN が 0 です。異常画像が 1 枚も無い、または 1 枚も検出していません。<br>TP + FP or TP + FN is zero: no abnormal images, or nothing detected. <!-- taj-row: nan-precision --> |

<!-- taj-unit: troubleshooting -->


## 完了チェックリスト / Completion checklist

**必須の項目だけ** を並べています。**演習 3（発展）は任意** で、未実施でも完了です。時間が余ったときだけ取り組んでください。

Only the **required** items are listed. **Exercise 3 (challenge) is optional**; leaving it undone still counts as complete. Do it only if you have spare time.

すべてにチェックが付いたら、このノートブックは完了です。

This notebook is complete when every box is checked.

- [ ] Part 1〜Part 5 を読み、すべてのコードセルを実行した<br>Read Parts 1-5 and ran every code cell
- [ ] 2 つの結果ファイルを読み込めた<br>Loaded both result files
- [ ] 同じ画像で比較表を作った<br>Built the comparison table on shared images
- [ ] しきい値を動かして FN / FP の変化を見た<br>Moved thresholds and observed FN/FP
- [ ] 比較の限界を 4 つ説明できる<br>Can explain the four limits of the comparison
- [ ] 演習 1〜2（必須）を実施した<br>Completed exercises 1-2 (required)
- [ ] 理解度チェックの 5 問に答えた<br>Answered the 5 questions
- [ ] ノートブックを保存した<br>Saved the notebook

<!-- taj-unit: completion-checklist -->


## さらに学ぶ / Further learning（任意 / optional）

<!-- taj-unit: optional -->

### 参考リンク / Reference links

- [Anomalib ドキュメント](https://anomalib.readthedocs.io/)
- [Ultralytics 検証モード](https://docs.ultralytics.com/modes/val/)
- [MVTec AD ベンチマーク](https://paperswithcode.com/sota/anomaly-detection-on-mvtec-ad)

<!-- taj-unit: reference-links -->

### 追加チャレンジ / Extra challenge（必須ではありません / not mandatory）

同じ画像に対して、YOLO で検出した位置と PatchCore のヒートマップの最大点がどれくらい離れているかを、3 枚で測ってみてください。

For three images, measure how far the YOLO box centre is from the brightest point of the PatchCore heatmap.

<!-- taj-unit: not-mandatory -->
